## Gold Layer - Trip Analysis

In [0]:
%run ./00_config

In [0]:
%sql

CREATE TABLE IF NOT EXISTS nyc_taxi_project.gold.trip_analysis
(
  window_start          TIMESTAMP,
  window_end            TIMESTAMP,
  pickup_borough        STRING,
  dropoff_borough       STRING,
  pickup_zone           STRING,
  dropoff_zone          STRING,
  is_rush_hour          BOOLEAN,
  is_airport_trip       BOOLEAN,
  total_trips           LONG,
  avg_trip_distance     DOUBLE,
  avg_duration_minutes  DOUBLE,
  avg_fare              DOUBLE,
  avg_tip_pct           DOUBLE,
  max_fare              DOUBLE
)
USING DELTA
COMMENT 'Gold 2 - Trip analysis by 1-hour tumbling window + watermark';

In [0]:
from pyspark.sql import functions as F

def process_gold_trip():

    query = (spark.readStream
                .format("delta")
                .table(SILVER2_TABLE)

             .withWatermark("tpep_pickup_datetime", "1 day")

             .groupBy(
                 F.window("tpep_pickup_datetime", "1 hour"),
                 "pickup_borough", "dropoff_borough",
                 "pickup_zone",    "dropoff_zone",
                 "is_rush_hour",   "is_airport_trip"
             )
             .agg(
                 F.count("*")                              .alias("total_trips"),
                 F.round(F.avg("trip_distance"), 2)        .alias("avg_trip_distance"),
                 F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
                 F.round(F.avg("fare_amount"), 2)          .alias("avg_fare"),
                 F.round(F.avg("tip_percentage"), 2)       .alias("avg_tip_pct"),
                 F.round(F.max("fare_amount"), 2)          .alias("max_fare")
             )
             .withColumn("window_start", F.col("window.start"))
             .withColumn("window_end",   F.col("window.end"))
             .drop("window")

             .writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", GOLD_TRIP_CHECKPOINT)
                .trigger(availableNow=True)
                .toTable(GOLD_TRIP))

    query.awaitTermination()

process_gold_trip()

In [0]:
%sql
-- Top 10 routes theo giờ cao điểm
SELECT
    window_start,
    pickup_zone,
    dropoff_zone,
    SUM(total_trips)                   AS total_trips,
    ROUND(AVG(avg_fare), 2)            AS avg_fare,
    ROUND(AVG(avg_duration_minutes),2) AS avg_duration,
    SUM(CASE WHEN is_airport_trip
        THEN total_trips ELSE 0 END)   AS airport_trips
FROM nyc_taxi_project.gold.trip_analysis
GROUP BY 1, 2, 3
ORDER BY total_trips DESC
LIMIT 10;